# Module 1 Note —— 量化分析入门

> 要理解量化，先要明白它的基础是什么，它优秀在哪里。

> 在金融领域，有三种基础的分析方式，分别为：**基本面分析**、**技术面分析**、**量化分析**。

> 我们先不要像经济学假设那样加上众多的限制，先只考虑 common sense。

---

## 笔记导览

**一、基础统计量**

return（回报）→ mean（均值）→ std（标准差）→ normalize（归一化）→ distribution（分布）→ 回报模型统一

**二、时间尺度与连续化**

时间对模型的影响 → 时间膨胀系数 $\alpha$ → 方差与时间的联系 → 离散模型（蒙特卡洛模拟）→ 连续模型（SDE）

**三、二叉树模型与期权定价**

搭建框架 → Δ对冲与套利 → 风险中性世界 → 期权定价通用公式 → 结合 SDE → 多重二叉树 → BS 方程

**四、随机微积分与伊藤引理**

构建布朗运动 → 条件期望与鞅 → 总二次变差 → 伊藤引理 → SDE 应用

## 一、基础统计量

### （一）return（回报）怎么算

假设 $S$ 为股票价格，$R$ 为回报（收益），那么按常识可得出：

$$R_i = \frac{S_{i+1} - S_i}{S_i}\tag{1}$$

这是最简单的回报计算公式。

### （二）mean（均值）怎么算

就是正常的数学公式（注意：下面的统计量均作用于**回报** $R_i$，而不是价格 $S_i$）：

$$\mu = \frac{\sum_{i=1}^{n} R_i}{n}\tag{2}$$

### （三）std（样本标准差）怎么算

就是正常的数学公式（注：此处 $n-1$ 为贝塞尔校正，减小偏差）：

$$\sigma = \sqrt{\frac{\sum_{i=1}^{n} (R_i - \mu)^2}{n-1}}\tag{3}$$

### （四）normalize（归一化）处理

最终要使得 mean $= 0$，std $= 1$，我们需要构建新变量便于比较和计算：

$$\Phi = \frac{R_i - \mu}{\sigma}\tag{4}$$

### （五）distribution（分布）构建

就用标准正态分布公式：

$$f(\Phi) = \frac{1}{\sqrt{2\pi}} e^{-\frac{\Phi^2}{2}}\tag{5}$$

到这里，我们说 $\Phi$ 是服从标准正态分布的一个随机变量。

### （六）回报模型的整理与统一

由式(4)整理，得：

$$R_i = \frac{S_{i+1} - S_i}{S_i} = \mu + \sigma \cdot Z, \quad Z \sim N(0,1)\tag{6}$$

其中 $Z$ 为标准正态变量，其实就是 $\Phi$。这里的 $\Phi$（即 $Z$）是模型的**随机噪声源**，它与任何其他资产、任何其他影响因素都完全独立、互不相关，这也是本模型成立的前提假设。

> 比如大宗商品，因为与其相关的东西太多，不易分离，就不适合这种数学建模。

## 二、时间尺度与连续化

### （一）时间对模型的影响

前面的模型显而易见是以天计的，我们把时间尺度缩小，先看一天：

$$\text{mean} = \mu \cdot \delta t$$

其中 $\mu$ 是年收益率（单位时间漂移率），$\delta t$ 就是一个极小的时间段（这里指一天），$\text{mean}$ 是平均日收益率。

在 $\delta t$ 足够小时，用"正常差值"计算回报与用"年化收益率差分"计算回报在**一阶近似**意义下相等（这是引入时间变量的必要一步联系）：

$$\bar{R} = \mu \cdot \delta t \quad \Longrightarrow \quad \frac{S_{i+1} - S_i}{S_i} = \mu \cdot \delta t\tag{7}$$

> 注：上式并非严格等式，而是 $\ln(1+x) \approx x$（即 $e^x \approx 1+x$）的一阶近似；$\delta t$ 越小误差越小（误差是 $\delta t$ 的二阶小量）。

于是：

$$S_{i+1} = S_i \cdot (1 + \mu \cdot \delta t)$$

令特定时间段 $t = M \cdot \delta t$，则由递归思想得：

$$S_M = S_0 \cdot (1 + \mu \cdot \delta t)^M$$

这正是复利思想的再次体现。

当时间步长足够小（$M \to \infty$）时，两边求极限（凑重要极限）：

$$S(t) = \lim_{M \to \infty} S_0 \left( 1 + \frac{\mu t}{M} \right)^M = S_0 \cdot e^{\mu t}\tag{8}$$

其实也可以表示成对数形式，对于 $\ln(1 + \mu \cdot \delta t)$，由泰勒展开近似为 $\mu \cdot \delta t$，细节不再赘述。

### （二）时间膨胀系数 $\alpha$

前面其实有个隐形假设：收益与时间成正比，即完整的写法 $\delta t^{\alpha}$ 中 $\alpha = 1$。若 $\alpha \neq 1$ 呢？（$\alpha$ 是时间膨胀系数）

若单步收益率为 $\mu \delta t^{\alpha}$，则经过 $M = t / \delta t$ 步之后：

$$S(t) = S_0 \left( 1 + \mu \delta t^{\alpha} \right)^{t / \delta t} \approx S_0 \, e^{\mu t \, \delta t^{\alpha - 1}}\tag{9}$$

即指数上会在 $\mu t$ 后多乘一个 $\delta t^{\alpha - 1}$，那就可以分类讨论了：

- 如果 $\alpha < 1$，$\delta t^{\alpha - 1}$ 是**负指数**幂。$\delta t$ 是个足够小的数，$\delta t \to 0$ 时指数 $\mu t \, \delta t^{\alpha - 1} \to +\infty$，对应的 $e^x \to +\infty$，现实意义就是**收益无穷大**（模型发散、失效）。

- 如果 $\alpha > 1$，$\delta t^{\alpha - 1}$ 是**正指数**幂。$\delta t \to 0$ 时指数 $\mu t \, \delta t^{\alpha - 1} \to 0$，对应的 $e^x \to 1$，现实意义就是**收益无穷小，买与不买一个样**。

只有 $\alpha = 1$ 才给出有限且非平凡的结果 $S_0 e^{\mu t}$——这就是标准模型假设收益与时间成正比的原因。

### （三）方差（标准差）与时间的联系

衡量收益，一个重要的东西就是方差（标准差的平方），它该如何与时间建立联系？

假设每天收益率的标准差均为 $0.01$（即每天方差为 $10^{-4}$），连续 5 天的标准差会是多少？

我们知道，对于独立事件，方差具有线性可加性：5 天总方差 $= 5 \times 0.01^2$，则 5 天标准差为：

$$\sqrt{5 \times 0.01^2} = \sqrt{5} \cdot 0.01$$

这正是时间膨胀系数 $\alpha = 1/2$ 的情况：**方差随时间线性增长，标准差随时间以平方根速度增长**（时间系数不可能为负）。即：

$$\text{Var}(\delta t) = \sigma^2 \cdot \delta t, \qquad \text{std}(\delta t) = \sigma \cdot \delta t^{1/2}\tag{10}$$

### （四）资产价格的离散模型

综上所有，分别代入变量得：

$$R_i = \mu \cdot \delta t + \sigma \cdot \Phi \cdot \delta t^{1/2} = \frac{S_{i+1} - S_i}{S_i}\tag{11}$$

这就是**资产价格的离散模型**。

整理一下：

$$S_{i+1} = S_i \cdot (1 + \mu \cdot \delta t) + \sigma \cdot S_i \cdot \Phi \cdot \delta t^{1/2}\tag{12}$$

式(12)表明：下一期资产价值等于趋势项与波动项之和。

**这就是蒙特卡洛模拟方法。**（每次模拟抽取一个 $\Phi \sim N(0,1)$，代入上式迭代，即可生成一条价格路径）

> 注：上式是"加法噪声"的简化离散化。更严谨的对数离散化是 $S_{i+1} = S_i \exp\left( \mu \delta t + \sigma \Phi \sqrt{\delta t} \right)$，两者在 $\delta t$ 很小、$\sigma \sqrt{\delta t}$ 不大时一阶近似相等。

### （五）从离散模型到连续模型（随机微分方程）

在离散模型的基础上，进一步将其推广到连续情形。

我们注意到，$\mu$ 和 $\sigma$ 乘上的 $\delta t$ 的指数并不相同：漂移项是 $\delta t$ 的**一阶**小量，波动项是 $\delta t$ 的**平方根阶**小量。当 $\delta t \to 0$ 时，$\delta t^{1/2} \gg \delta t$，所以**极短时间内波动项占主导**；而随着时间拉长，漂移项 $\mu t$ 线性累积，最终**长期趋势占主导**——这就是"短期波动优先、长期趋势优先"的底层数学原理。

$$S_{i+1} - S_i = \mu \cdot S_i \cdot \delta t + \sigma \cdot S_i \cdot \Phi \cdot \delta t^{1/2}$$

将时间尺度取得无穷小，令 $\delta t \to 0$：

$$dS = S_{i+1} - S_i, \qquad dX = \Phi \cdot \sqrt{dt}$$

右边就把 $S_i$ 直接写作 $S$，就变成了：

$$dS = \mu S \, dt + \sigma S \, dX\tag{13}$$

**这个，就是 SDE（随机微分方程）。**

其中 $dX$ 就是随机微分（一个随机变量！不是黎曼积分中确定的某值）。注意它的分布：

$$E[dX] = 0, \quad E[(dX)^2] = dt, \quad \text{Var}[dX] = dt\tag{14}$$

即 $dX \sim N(0, dt)$——它**不服从标准正态分布**（$dX / \sqrt{dt}$ 才是 $N(0,1)$）。更准确地说，$dX = X_{t+dt} - X_t$ 是布朗运动 $X_t$ 的**增量**：均值为 0，即长期来看对资产的变动趋势无影响；方差为 $dt$，即在极小的时间会对资产造成影响，造成的影响也是完全随机的。

随机微积分弥补了以前金融建模的动态部分的缺憾，成功将模型更细致地映射到了现实世界中。

> asset（资产）是一种统称，对很多商品都适用。其并非适用于所有，比如利率，虽然可以用，但就是不匹配。

> 这只是定价用的，不适用于风控等等。

> 所有不严谨的东西都将后续补充。

## 三、二叉树模型与期权定价

> **二叉树模型（随机游走的极端简化模型）——期权定价**

> 能够体现金融的两大核心思想：**对冲**和**无套利**，同时能够给出期权定价。

> 能通过它简化复杂的金融思想。

### （一）搭建一个简单的框架

我们先搭建一个非常简单的框架作为后续分析的基础：

- 一支股票 a stock
- 一个看涨期权 a call option tomorrow（欧式期权，到期才可行权）
- 涨跌幅已知
- 利率 $r = 0$（未来的折现就不用考虑利率了）

股价 $S = 100$：

- 概率 $p = 0.6$，上涨到 $101$
- 概率 $p = 0.4$，下降到 $99$
- 时间为 1 天

同时引入一份看涨期权：

$$V = \max\{S_T - K, 0\}\tag{15}$$

行权价 $K = 100$。如果上涨，作为买方（long 方），肯定行权，挣了 1 元；下跌肯定不行权，挣了 0 元。（无期权费）

基于上述信息，应如何确定期权当前的合理价格？

一般我们会认为：期权价格是期望值：

$$V = \frac{1 \times 0.6 + 0 \times 0.4}{1 + r} = 0.6$$

（$1 + r$ 为折现因子）

注意：这是用**真实概率**求期望的朴素定价，其中**不包含风险溢价**；现实定价必然包含风险溢价，即：

$$V = \frac{1 \times 0.6 + 0 \times 0.4}{1 + r + \text{risk}}$$

风险溢价的大小因人而异；但分母增大必使估值变小，故合理价格一定 $< 0.6$。

要处理 risk，两种方法：**对冲**和**风险中性**。

### （二）Δ对冲：构建投资组合

考虑由 1 份期权与 $\frac{1}{2}$ 份股票构成的组合（配置比例的缘由见后文）。

那么 0 时刻成本（投资组合价值）是 $V - \frac{1}{2} \times 100$（买期权花了 $V$ 元，又卖出 $\frac{1}{2}$ 份股票获得 50 元，也就是做空）。

由上述可知，资产组合的价值为：

$$S^{+} = 1 - \frac{1}{2} \times 101 = -49.5$$

$$S^{-} = 0 - \frac{1}{2} \times 99 = -49.5$$

即股票价格不论上涨或下跌，最终价值一样，上涨或下跌的风险就被完全抵消了。

既然明日组合总价值确定为 $-49.5$，今日价值应如何推算？

由于 $r = 0$ 时无风险资产的现值等于终值，故今日组合价值必然同为 $-49.5$。

即使将资金闲置（放在枕下也不会增值），无套利也要求组合终值等于现值；否则便存在承担极小风险而获取无风险回报的套利机会。

由于市场在不断的调节，有套利空间人们就会蜂拥而至，导致利差被抹平，最终依旧是无套利。所以无风险价格就一定是最终价格。

综上算得，$V - 50 = -49.5$，即 $V = 0.5$ 才是最好的。

值得注意的是，真实概率 $p$ 在计算中完全没有被用到，由此得到一个反直觉的结论：**期权定价与股票的真实涨跌概率无关**，只取决于涨跌幅度（波动率）。（涨跌风险已被对冲消除）

我们其实应该关注涨跌幅度（波动率），因为波动越大，期权价格越高。

#### 1. 有套利空间，怎么套利？我们算出的价格为什么一定是最终价格？

假设市场上实际期权价格 $< 0.5$，我们算出理论定价 $0.5$，证明现在在市场上买期权非常划算（比预计少花钱了），继续上述过程获得利润。

假设市场上实际期权价格 $> 0.5$，我们算出理论定价 $0.5$，证明现在在市场上卖期权非常划算（能卖更多的钱），继续上述过程获得利润。

**举例一**：$V = 0.6$，$S = 100$（市场价高于理论价 → 卖出期权并买入股票）

<p align="center">表1&nbsp;&nbsp;卖出期权、买入股票组合的套利现金流（市场价高于理论价）</p>

| 时刻 | 操作 | 现金流 |
| --- | --- | --- |
| $t = 0$ | 卖期权（空头） | $+0.6$ |
| $t = 0$ | 买股票（多头） | $-\frac{1}{2} \times 100$ |
| $t = 0$ | 期初净现金流 | $-49.4$ |
| 涨（$S = 101$） | short 方赔付 | $-1$ |
| 涨（$S = 101$） | 卖出股票 | $+\frac{1}{2} \times 101$ |
| 涨（$S = 101$） | 期末净现金流 | $+49.5$ |
| 跌（$S = 99$） | short 方赔付 | $+0$ |
| 跌（$S = 99$） | 卖出股票 | $+\frac{1}{2} \times 99$ |
| 跌（$S = 99$） | 期末净现金流 | $+49.5$ |

综上无论如何都是赚到了 $49.5 - 49.4 = 0.1$ 元。

**举例二**：$V = 0.4$，$S = 100$（市场价低于理论价 → 买入期权并卖空股票）

<p align="center">表2&nbsp;&nbsp;买入期权、卖空股票组合的套利现金流（市场价低于理论价）</p>

| 时刻 | 操作 | 现金流 |
| --- | --- | --- |
| $t = 0$ | 买期权（多头） | $-0.4$ |
| $t = 0$ | 卖股票（空头） | $+\frac{1}{2} \times 100$ |
| $t = 0$ | 期初净现金流 | $+49.6$ |
| 涨（$S = 101$） | long 方行权 | $+1$ |
| 涨（$S = 101$） | 买回股票 | $-\frac{1}{2} \times 101$ |
| 涨（$S = 101$） | 期末净现金流 | $-49.5$ |
| 跌（$S = 99$） | long 方不行权 | $+0$ |
| 跌（$S = 99$） | 买回股票 | $-\frac{1}{2} \times 99$ |
| 跌（$S = 99$） | 期末净现金流 | $-49.5$ |

综上无论如何都是赚到了 $49.6 - 49.5 = 0.1$ 元。

综上，两个例子中套利的本质相同：**用期权与股票构造与股价涨跌无关的"锁定"组合**，使期末组合价值确定，期初即可赚取确定的价差。

> 要知道，现实生活中肯定会有更多因素，比如滑点，情绪化交易等。

#### 2. 为什么是配置 $\frac{1}{2}$ 的股票？不能是其他数字吗？

设 $\Delta$ 为对冲比率，表示每持有一份期权，需卖出标的资产（一般指股票）的份数，反映了期权价格对于股票价格变化的敏感度，其离散形式即：

$$\Delta = \frac{\partial V}{\partial S}$$

那么根据上面的框架，我们得出：

- 初始成本：$V - \Delta \times 100$
- 股价上涨：$1 - \Delta \times 101$
- 股价下跌：$0 - \Delta \times 99$

要保证股价无论上涨还是下跌，最终收益不变，则上涨和下跌价值必然相等：

$$1 - 101\Delta = 0 - 99\Delta \quad \Longrightarrow \quad \Delta = 0.5$$

推广一下：

$$\Delta = \frac{\text{期权收益范围}}{\text{股价变化范围}} = \frac{V^{+} - V^{-}}{S^{+} - S^{-}}$$

注意：对冲是宽泛的，但是 Δ对冲一般指期权和股票的组合。

#### 3. 无风险回报率（利率）$r \neq 0$，怎么办？

只需将未来价值折现到现在。注意：一年按 252 个交易日计，，所以 $r = \frac{0.1}{252}$（此处 $0.1$ 是年利率）。

#### 4. 现在期权定价基本结束了，我们可以反思一下，股票价格真的合理吗？

之前用真实概率算得未来股价期望是 $0.6 \times 101 + 0.4 \times 99 = 100.2$，现在定价 $100$，相当于去掉了 $0.2$ 的风险补偿，其实是非常合理的。

**完全市场**：非常理想的市场。没有税收、资产可无限分割等乌托邦式市场。

在完全市场里，我们说：

$$V - \Delta \cdot S = \text{RFA (Risk Free Asset)}\tag{16}$$

就是资产组合的价值等于无风险资产的价值。

所以期权收益也就是投资收益与无风险资产收益的总和，或者说期权是完全由 payoff 决定的；难以计算期权价格时，也可以计算标的资产价值来估算期权价值<sup>1</sup>。

### （三）真实世界与风险中性世界

<p align="center">表3&nbsp;&nbsp;真实世界与风险中性世界的对比</p>

| | 真实世界 | 风险中性世界 |
| --- | --- | --- |
| 风险态度 | 人对风险很敏感，风险越高回报越大 | 人承认风险但不关心风险 |
| 风险溢价 | 人没有办法统一风险溢价，所以没有办法用期望定价 | 人不承认风险溢价 |
| 定价方法 | 人可以通过风险对冲来定价 | 人可以通过无风险回报率来直接定价 |

怎样从真实世界转到风险中性世界？

还是刚才的框架，但风险中性世界中概率不一样了（注意这里 $r = 0$，故不必折现）：

$$E[S] = p' \times 101 + (1 - p') \times 99 = 100, \quad 得 \quad p' = 0.5$$

则风险中性概率 $p' = 0.5$，这样就可以保证风险中性世界中 $t = 0$ 时刻股票定价与真实世界定价完全一样。

期权价格怎么算？这时候可以用期望了，因为风险中性世界以无风险利率折现：

$$V = \frac{0.5 \times 1 + 0.5 \times 0}{1 + r} = 0.5$$

与之前在真实世界的定价完全相同！就算有 $r$，也好算，折现就行了。

好的，这样的话就两种方法都讲完了，显然他们殊途同归，那怎么联合起来呢？

设 $u$ 为上涨因子，则 $S^{+} = u \cdot S$，$u > 1$；

设 $v$ 为下跌因子，则 $S^{-} = v \cdot S$，$0 < v < 1$；

真实世界概率为 $p$，时间段（时间步长）为 $\delta t$。

### （四）期权定价通用公式：二叉树模型

#### 1. 将 Δ对冲与风险中性世界结合起来

对于某一投资组合：

$$\Pi = V - \Delta \cdot S\tag{17}$$

由上述假设：

$$\Delta = \frac{V^{+} - V^{-}}{(u - v) \cdot S}\tag{18}$$

再把 $\Delta$ 反代入原来的上涨或下跌的投资组合公式，得：

$$\Pi^{+} = V^{+} - \frac{u \cdot (V^{+} - V^{-})}{u - v}$$

$$\Pi^{-} = V^{-} - \frac{v \cdot (V^{+} - V^{-})}{u - v}$$

如果有无风险回报率 $r$ 呢——提出**折现因子**概念。

首先，由上面的时间对资产收益率的影响 $S(t) = S_0 \cdot e^{\mu t}$ 得：

$$\Pi_{\text{future}} = \Pi_{\text{now}} \cdot e^{\mu t}$$

那么对于现在来说：

$$\Pi_{\text{now}} = \frac{1}{e^{\mu t}} \cdot \Pi_{\text{future}}$$

由于组合无风险，收益率 $\mu$ 就是无风险利率（收益率）$r$；时间段 $t$ 就是 $\delta t$。

根据泰勒展开，在 $\delta t$ 足够小时，$e^{\mu t} \approx 1 + r \cdot \delta t$，所以折现因子就是：

$$\frac{1}{1 + r \cdot \delta t}\tag{19}$$

在无风险情况下：

$$V - \frac{V^{+} - V^{-}}{u - v} = \frac{1}{1 + r \cdot \delta t} \cdot \left( V^{+} - \frac{u \cdot (V^{+} - V^{-})}{u - v} \right)$$

注：代 $\Pi^{+}$ 和 $\Pi^{-}$ 都行，反正都相等。

故：

$$V = \frac{V^{+} - V^{-}}{u - v} + \frac{1}{1 + r \cdot \delta t} \cdot \frac{u \cdot V^{-} - v \cdot V^{+}}{u - v}$$

再整理一下：

$$V = \frac{1}{1 + r \cdot \delta t} \cdot \left[ p' \cdot V^{+} + (1 - p') \cdot V^{-} \right], \quad p' = \frac{1 + r \cdot \delta t - v}{u - v}\tag{20}$$

#### 2. 结合 SDE：确定 $u$、$v$、$p$

我们知道在 SDE 里，重要的参数是 $\mu$ 和 $\sigma$；而在二叉树模型里，参数是 $u$、$v$、$p$（真实概率）。我们必须在它们之间建立关系。

根据 SDE $dS = \mu S \, dt + \sigma S \, dX$ 得：在这样足够小的时间段 $\delta t$ 中，均值的变化量其实是 $\mu S \delta t$。

在二叉树模型中，我们要计算这样的变化量，就是 $\delta t$ 时间后股价的期望减去最初的股价。列式为：

$$\left( p \cdot u \cdot S + (1 - p) \cdot v \cdot S \right) - S = \mu S \delta t$$

得不出来什么，我们还需要其它式子。

因为（$dS$ 的方差来自波动项：漂移项 $\mu S dt$ 是确定性的，方差为 0）：

$$\text{Var}(dS) = \text{Var}(\mu S dt) + \text{Var}(\sigma S dX) = 0 + \sigma^2 S^2 \text{Var}(dX) = \sigma^2 S^2 dt$$

相应的，二叉树模型的方差为：

$$\text{Var}(S) = p \cdot \left[ uS - E(S) \right]^2 + (1 - p) \cdot \left[ vS - E(S) \right]^2$$

其中 $E(S) = p \cdot u \cdot S + (1 - p) \cdot v \cdot S$。

最后令 $\text{Var}(dS) = \text{Var}(S)$（方差匹配）。

把 $u$、$v$、$p$ 看成三个未知数；把 $\mu$ 和 $\sigma$ 看成两个参数。这样三个未知数，两个方程，有无数个解。

在金融领域，为简便运算，取一组合适的解：令 $u + v = 2$（涨跌幅度对称），并要求 $uv \approx 1$（**一阶近似**：精确地 $uv = 1 - \sigma^2 \delta t$，与 1 只差二阶小量，可忽略）。于是取：

$$u = 1 + \sigma \sqrt{\delta t}, \quad v = 1 - \sigma \sqrt{\delta t}\tag{21}$$

代入上面第一个方程简化后得：

$$p = \frac{1}{2} + \frac{\mu}{2 \sigma}\sqrt{\delta t}\tag{22}$$

另外我们可以把 $u$、$v$ 直接带入之前的 $p'$ 公式，化简后居然得到了：

$$p' = \frac{1}{2} + \frac{r}{2 \sigma}\sqrt{\delta t}\tag{23}$$

也就是说，风险中性概率与真实概率的不同点在于收益率上：真实世界是实际收益率 $\mu$，风险中性世界是无风险收益率 $r$。

从这里我们也可以知道，**期权的价值就是在风险中性世界中未来价值期望的折现**<sup>2</sup>

### （五）多重二叉树

把单步二叉树重复 $N$ 期，就得到多重二叉树：

- 涨 $k$ 次、跌 $N-k$ 次的终端价格为 $S_0 u^{k} v^{N-k}$；由于 $v < 1$，上方路径价格越来越大，下方路径价格越来越小、趋近于 0。

- 关键性质：乘法可交换，$uvS = vuS$ **恒成立**，所以"先涨后跌"与"先跌后涨"必然到达同一节点——树是**可重组的**（recombining tree）。这正是"上下增长幅度取对称"的用意：节点数从 $2^N$ 条路径压缩为 $\frac{(N+1)(N+2)}{2}$ 个，指数级计算量降为平方级。

它有什么用？**欧式期权定价**。从期末开始逐层倒推：期末每个节点按 $V_T = \max\{S_T - K, 0\}$ 定值，

然后自后向前逐层应用定价公式[式(20)]，即可倒推得到 $t = 0$ 时刻的期权价值。

我们是时候从离散扩展到连续了。

### （六）从二叉树到连续：BS 方程

先回忆定价模型[式(20)]，对 $V^{+}$、$V^{-}$ 分别写出其在节点处的取值：



$$V^{+} = V(uS, \, t + \delta t) = V\left( (1 + \sigma \sqrt{\delta t}) \cdot S, \, t + \delta t \right)$$

泰勒展开得：

$$V^{+} = V(S, t) + \frac{\partial V}{\partial t} \delta t + \frac{\partial V}{\partial S} \sigma S \sqrt{\delta t} + \frac{1}{2} \frac{\partial^2 V}{\partial S^2} \sigma^2 S^2 \delta t + o(\delta t)\tag{24}$$

以及：

$$V^{-} = V(vS, \, t + \delta t) = V\left( (1 - \sigma \sqrt{\delta t}) \cdot S, \, t + \delta t \right)$$

泰勒展开得：

$$V^{-} = V(S, t) + \frac{\partial V}{\partial t} \delta t - \frac{\partial V}{\partial S} \sigma S \sqrt{\delta t} + \frac{1}{2} \frac{\partial^2 V}{\partial S^2} \sigma^2 S^2 \delta t + o(\delta t)\tag{25}$$

把 $p' = \frac{1}{2} + \frac{r}{2 \sigma}\sqrt{\delta t}$ 代入进去，忽略 $o(\delta t)$，且令 $\frac{1}{1 + r \cdot \delta t} \approx 1 - r \cdot \delta t$，这个 $V(S, t)$ 方程就会变成：

$$V(S,t) = (1 - r \delta t) \cdot \left[ V(S,t) + \frac{\partial V}{\partial t} \delta t + \frac{1}{2} \frac{\partial^2 V}{\partial S^2} \sigma^2 S^2 \delta t + \frac{r}{2 \sigma}\sqrt{\delta t} \cdot 2 \cdot \frac{\partial V}{\partial S} \sigma S \sqrt{\delta t} \right]$$

$$= V - rV \delta t + \frac{\partial V}{\partial t} \delta t + \frac{1}{2} \frac{\partial^2 V}{\partial S^2} \sigma^2 S^2 \delta t + r S \frac{\partial V}{\partial S} \delta t - \frac{1}{2} \frac{\partial^2 V}{\partial S^2} r \sigma^2 S^2 \delta t^2 - r^2 S \frac{\partial V}{\partial S} \delta t^2$$

忽略 $o(\delta t)$，再消掉 $\delta t$ 得：

$$\frac{\partial V}{\partial t} + \frac{1}{2} \sigma^2 S^2 \frac{\partial^2 V}{\partial S^2} + r S \frac{\partial V}{\partial S} - r V = 0\tag{26}$$

**这就是 BS 方程。**

其中各项含义：

- $\frac{\partial V}{\partial t}$：期权随时间的变化率
- $\frac{1}{2} \sigma^2 S^2 \frac{\partial^2 V}{\partial S^2}$：波动率对期权价值的影响
- $r S \frac{\partial V}{\partial S}$：期权价值的无风险增长
- $r V$：期权价值的无风险折现

这样完整的方程的实际意义是：

> 期权的时间价值 + 波动率对期权价值的影响 + 期权价值的无风险增长 = 期权价值的无风险折现

注：其实这个推导在数学上不是完全严谨，后续也会有补充。

## 四、随机微积分与伊藤引理

### （一）构建布朗运动（BM）：理解时间序列数据的动态变化

#### 1. 从抛硬币开始

先从抛硬币这一简单例子说起：，正面 $+1$ 元，反面 $-1$ 元。

对这个事件，先做一些基础设置（$R_i$ 表示第 $i$ 次抛硬币的收益）：

1. $E[R_i] = 0$：对于抛硬币这一事件，第 $i$ 次获得收益的期望为 0。
2. $E[R_i^2] = 1$：实际结果不是 $+1$ 就是 $-1$，平方后一定是 1，期望也就是 1；或者说波动是一定的，方差也就是 1。
3. $E[R_i \cdot R_j] = 0 \ (i \neq j)$：说明抛硬币是一个独立事件，两次事件相关性为 0：

$$\rho_{ij} = \frac{\text{cov}(R_i, R_j)}{\sigma_{R_i} \cdot \sigma_{R_j}} = \frac{E[R_i \cdot R_j] - E[R_i] \cdot E[R_j]}{\sigma_{R_i} \cdot \sigma_{R_j}}$$

#### 2. 累积收益 $S_i$ 及其性质

设 $S_i$ 是前 $i$ 次抛硬币的总收益：

$$S_i = \sum_{j=1}^{i} R_j$$

它的特性有：

$$E[S_i] = E[R_1 + R_2 + \cdots + R_i] = E[R_1] + E[R_2] + \cdots + E[R_i] = 0$$

$$E[S_i^2] = \text{Var}(S_i) + (E[S_i])^2 = i \times 1 + 0 = i$$

也可直接展开理解：由独立性，"两事件乘积"项的期望均为 0，仅余 $i$ 个自身平方项，故总值为 $i$。

#### 3. 条件期望与无条件期望

例如，已知抛了 5 次硬币，第 6 次的期望应如何计算？

这里需要区分两类期望：**条件期望**与**无条件期望**。

先计算无条件期望：

$$E(R_i) = 1 \times 0.5 + (-1) \times 0.5 = 0$$

那么（期望的线性可加性）：

$$E(S_6) = E(R_1) + E(R_2) + E(R_3) + E(R_4) + E(R_5) + E(R_6) = 0$$

条件期望则须给定前提条件。设已知 $S_5 = 3$：

$$E[S_6 \mid S_5 = 3] = E(R_6 \mid S_5 = 3) + E(S_5 \mid S_5 = 3) = 0 + 3 = 3$$

#### 4. 测度论的基本概念

由上述计算可抽象出三个基本要素：

1. 一个**样本空间** $\Omega$
2. 一个**信息集（filtration）**：$\mathcal{F}$（随时间展开的信息流）
3. 一个**概率测度** $P$

**P 测度（物理测度 physical measure，即真实世界概率）**：可以举二叉树模型为例，上涨 0.6、下跌 0.4 就是 P 测度。

**Q 测度（风险中性概率）**：风险中性世界的概率，与 P 测度不同。

也就是说，两种测度下，事件的发生与否一定相同，只有概率被调整了。这说明 P 与 Q 是**等价测度**。

#### 5. 可测性与条件期望的性质

若随机变量 $X$ 由信息集 $\mathcal{F}$ 唯一确定（adapted to / measurable w.r.t. $\mathcal{F}$），则称"$X$ 对 $\mathcal{F}$ 可测"。

也就是说，$X$ 完全由信息集 $\mathcal{F}$ 决定，就说 $X$ 在 $\mathcal{F}$ 里可测。更一般地，我们常考虑对未来变量 $X_n$ 在更早信息集 $\mathcal{F}_m$ 下的条件期望（$n > m$）：

$$E[X_n \mid \mathcal{F}_m], \quad n > m$$

条件期望满足以下性质：

1. **线性性**：

$$E[\alpha X + \beta Y \mid \mathcal{F}] = \alpha E[X \mid \mathcal{F}] + \beta E[Y \mid \mathcal{F}]$$

2. **塔性质（tower property）**：对于信息集 $\mathcal{F} \subseteq \mathcal{G}$，

$$E[E[X \mid \mathcal{G}] \mid \mathcal{F}] = E[X \mid \mathcal{F}]\tag{27}$$

（最小的信息集决定最终结果）

3. **无信息集特性（由 2 推导而来）**：对于一系列信息集，若已知某一信息集为空集 $\emptyset$，则它必然是最小信息集。也就是说：

$$E[E[X \mid \mathcal{F}] \mid \emptyset] = E[X]$$

这是塔性质的特例。

> 举例：我们已知有过去 5 年的股价数据，突然被人告知最近 3 年的数据都是无效的。那么对于这 3 年来说，我们计算股价就是计算无条件期望。

4. 如果 $X$ 对 $\mathcal{F}$ 可测，那么 $E[X \mid \mathcal{F}] = X$：$X$ 在 $\mathcal{F}$ 信息集里完全已知，算它的期望其实就是得出 $X$ 本身。

5. 如果 $X$ 对 $\mathcal{F}$ 可测，那么 $E[XY \mid \mathcal{F}] = XE[Y \mid \mathcal{F}]$（性质 4 的推广）。

6. 如果 $X$ 与 $\mathcal{F}$ 独立（或者说 $\mathcal{F}$ 对 $X$ 是无效信息），那么 $E[X \mid \mathcal{F}] = E[X]$。

7. 如果 $X \geq 0$，则 $E[X \mid \mathcal{F}] \geq 0$。

8. **Jensen 不等式**：设 $f$ 为开口向上的凸函数，显然对于该函数，左右任取两点构成的弦上的点必然不低于函数图像上的点。设 $\lambda \in [0, 1]$，就有：

$$f(\lambda x_1 + (1 - \lambda) x_2) \leq \lambda f(x_1) + (1 - \lambda) f(x_2)\tag{28}$$

扩展到期望上，我们可以提取它的思想：**函数值的期望，必然不小于其自变量期望所对应的函数值**，即：

$$f(E[X \mid \mathcal{F}]) \leq E[f(X) \mid \mathcal{F}]\tag{29}$$

#### 6. 马尔可夫性质与鞅

综上，回到最初的例子，正是 $E[S_6 \mid R_1, R_2, R_3, R_4, R_5] = S_5$。

从这里看：基于过去的信息，未来的期望等于已有的累计值（历史路径）。由此引出以下两个重要性质：

1. **马尔可夫性质**：对于某一随机变量 $S_i$，在给定全部历史信息的条件下，它的概率分布只取决于 $S_{i-1}$（当前状态），其余历史数据无用。

   - 注意：我们只约定当前值依赖于先前值，并不一定当前值与先前值相等。换句话说，即使未来是上涨或下跌，只要是认为当前值依赖于先前值，都满足马尔可夫性质。
   - 刚才的例子里，$E[S_6 \mid R_1, R_2, R_3, R_4, R_5] = S_5$ 一式正体现了马尔可夫性质。

2. **鞅过程（martingale）**：马尔可夫性质的特例。在给定全部历史信息的条件下，未来任意时刻随机变量 $S_i$ 的条件期望，就等于当下变量 $S_j$ 的值：

$$E[S_i \mid S_j, \ j < i] = S_j\tag{30}$$

看一下两者之间的联系与区别吧：

- 马尔可夫性质认为：未来的事件只依赖于现在的事件，与过去怎样无关，但未来的值不确定。
- 鞅过程认为：基于马尔可夫性质，未来时刻的条件期望严格等于当前事件的值。

#### 7. 总二次变差与布朗运动

在上述概念的基础上，引入**总二次变差（Quadratic Variation, QV）**：

$$QV_i = \sum_{j=1}^{i} (S_j - S_{j-1})^2\tag{31}$$

反映的是收益变化的平方和，体现了函数的波动程度。

联系前面的结果（每步 $E[(\Delta S_j)^2] = \text{Var}(R_j) = 1$）：

$$E[S_i^2] = \text{Var}(S_i) = \sum_{j=1}^{i} E[(S_j - S_{j-1})^2] = i$$

回到二次变差与时间的联系。假设在有限时间 $t$ 内抛 6 次硬币，时间等分，每次收益 $S_j - S_{j-1}$ 为 $\sqrt{t/6}$。显然这样算出来的总二次变差为 $6 \times (t/6) = t$，这样就与时间建立了联系。抛 $n$ 次，结论还是一样的。

当交易次数 $n \to \infty$ 时，每次交易的时间（时间步长）趋近于 0。在这种极限情况下的随机游走，就是**布朗运动**。

相应地，假设初始值为 0，显然：

$$E[S(t)] = 0, \qquad E[S(t)^2] = t, \qquad \text{Var}(S(t)) = t\tag{32}$$

#### 8. 布朗运动的性质

1. **有限性（增量与 $\sqrt{\Delta t}$ 成正比）**：布朗运动的增量必然与时间步长的平方根成正比。

   为什么？通过刚才的例子，时间步长 $\Delta t = t / n$，$QV = \sum_{i=1}^{n} (\Delta S_i)^2$。我们姑且用反证法先理解一下，正统的证明暂时不谈：

   - ①我们说收益是 $+C$（常数）。此时 $QV = n \cdot C^2$。显然当交易次数 $n \to \infty$ 时，$QV \to +\infty$：当收益与时间无关时，收益发散到无穷，不可能是标准布朗运动。
   - ②我们说 $\Delta S_i = (\Delta t)^k \ (k \neq 1/2)$，此时 $QV = t^{2k} / n^{2k-1}$：
     - 如果 $k > 1/2$，则 $2k - 1 > 0$。当交易次数 $n \to \infty$ 时，$QV \to 0$：这样的话收益趋近于 0，不可能是标准布朗运动。
     - 如果 $k < 1/2$，则 $2k - 1 < 0$ 恒成立。当交易次数 $n \to \infty$ 时，$QV \to +\infty$：这样的话收益发散到无穷，不可能是标准布朗运动。
   - 只有 $k = 1/2$，即增量与 $\sqrt{\Delta t}$ 成正比，$QV$ 才收敛到一个有限且非零的值 $t$。

2. **连续性**：路径是连续的（没有跳跃）。

3. **马尔可夫性**：见上文第 6 小节。

4. **鞅性质**：见上文第 6 小节。

5. **二次变差**：布朗运动的总二次变差几乎必然收敛于有限时间 $t$；而一般光滑函数的 $QV \to 0$。

6. **正态性**：连续布朗运动是离散随机游走时间步长无穷小的极限情况。将 $X$ 视作某个时间点 $t$ 的瞬时随机状态，$dX$ 是 $X$ 的一个增量，那么布朗运动的增量部分 $dX \sim N(0, dt)$。

### （二）SDE 与伊藤引理

在 SDE 里，有两个变量：时间 $t$ 和布朗运动 $X$，所以在求导时要注意分别求偏导。

#### 1. 单变量情形（直觉推导）

对于某一函数 $F(X)$，将 $dX$ 看作其自变量的增量，那么根据泰勒展开，就有：

$$F(X + dX) = F(X) + \frac{dF}{dX} dX + \frac{1}{2} \frac{d^2 F}{dX^2} (dX)^2 + o((dX)^2)$$

令 $F(X + dX) - F(X) = dF$，那么：

$$dF = \frac{dF}{dX} dX + \frac{1}{2} \frac{d^2 F}{dX^2} (dX)^2 + o((dX)^2)$$

我们把 $(dX)^2$ 替换成 $dt$（不是严格相等，后续再补充严谨证明），那就有：

$$\text{Ito 1:} \qquad dF = \frac{dF}{dX} dX + \frac{1}{2} \frac{d^2 F}{dX^2} dt\tag{33}$$

#### 2. 双变量情形（完整的伊藤引理）

前面说的是隐式形式，我们确定一下变量 $X$、$t$：

$$dF(X, t) = \frac{\partial F}{\partial X} dX + \frac{\partial F}{\partial t} dt + \frac{1}{2} \frac{\partial^2 F}{\partial X^2} (dX)^2 + o(dt)$$

简单得出：

$$\text{Ito 2:} \qquad dF(X, t) = \frac{\partial F}{\partial X} dX + \left( \frac{\partial F}{\partial t} + \frac{1}{2} \frac{\partial^2 F}{\partial X^2} \right) dt\tag{34}$$

### （三）SDE 的应用

通过以上推导，我们得出了 SDE 的一般形式：

$$dS = f(S, t) \, dt + g(S, t) \, dX\tag{35}$$

其中 $f(S, t)$ 叫**漂移项**，$g(S, t)$ 叫**波动项**。

#### 1. 股票的 SDE 模型

由式(35)的一般形式中取 $f(S,t) = \mu S$、$g(S,t) = \sigma S$，即得：

满足几何布朗运动，避免了负值的产生。其中 $dS / S$ 就是股票的瞬时回报率。

#### 2. 短期利率的 SDE 模型（CIR 模型）

$$dr = \mu (\theta - r) \, dt + \sigma \sqrt{r} \, dX\tag{36}$$

均值回归（mean reversion）：其中 $\mu$ 是回归速度，$\theta$ 是均衡利率水平（若把均值回归项写成 $(v - \mu r)\,dt$ 的形式，则 $\theta = v/\mu$）；$\sqrt{r}$ 项在利率水平低时减小了波动幅度，避免了负值的大量产生。

#### 3. SDE 的积分形式

对于任意资产 $W$，把 $dW = f(W, t)\,dt + g(W, t)\,dX$ 两边从 0 积分到 $t$：

$$W(t) = W(0) + \int_0^t f(W_\tau, \tau) \, d\tau + \int_0^t g(W_\tau, \tau) \, dX(\tau)\tag{37}$$

其中第一项是普通（黎曼）积分，第二项 $\int_0^t g \, dX$ 是**伊藤积分**（随机积分，$dX(\tau)$ 是随机微分而不是确定量）。

> 注：这里漂移项 $f$ 对应 $d\tau$、波动项 $g$ 对应 $dX$，与前面一般形式 $dS = f\,dt + g\,dX$ 保持一致。